# FieldTools Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pjmartel/FieldTools/blob/main/FieldTools.ipynb)

---
FieldTools.py can be used to calculate electric fields from MD trajectories. The script requires standard MD **trajectory** and **parameter** files as input.
Furthermore a **target** file needs to be provided that specifies the positions at which the field will be calculated.

---     
<font color="red">**Attention: Trajectories must be imaged (or use -pbc True)!**</font>  
<font color="red">FieldTools calculates the fields from the exact location of all atoms in the simulation without considering periodicity, unless -pbc True is used.</font>

---
**Requirements**
- Python3.X
- numpy
- MDAnalysis (install using pip install mdanalysis) or pytraj

---
**Usage**

    python utils/FieldTools.py -nc <trajectory file>              
                               -parm <parameter file>             
                               -target <target file>              
                               [-solvent <non-protein residues>] 
                               [-exclude_atoms <exclusion list>]  
                               [-TIP4P <True|False>]               
                               [-verbose <True|False>]             
                               -out <output file> 

- Trajectory (-nc) and parameter (-parm) input file:
  - The script uses MDAnalysis to load the trajectory and thus accepts any file types accepted by MDAnalysis.
  - Amber: .parm7/.prmtop topology with .nc/.mdcrd trajectory.
  - GROMACS: .tpr (recommended) or .top topology with .xtc/.trr trajectory. Use -pbc True for GROMACS trajectories, whose molecules are usually broken over the periodic boundaries.
- A -target file that specifies the positions at which the field is to be calculated:
  - FieldTools can calculate the field either along a bond or at a specific atom.
  - For field calculations along bonds, two target atoms need to be defined.
  - For field calculations at a specific point, only one target atom needs to be defined.
    The result is the magnitude of the field vector |E|; use -vector_out to save the vectors.
  - Multiple targets can be specified and calculated in parallel.
- A -solvent flag that specifies all solvent (and ion!) molecules (default: auto, the common water and ion names found in the system):
  - For the protein, FieldTools will give the field of each individual protein residue.
  - Solvent atoms will not be included in the protein field, but summed up separately in the solvent field.
  - The script will furthermore provide separate fields for each type of solvent molecule.
- Optional -exclude_atoms flag that defines which atoms will not be included in the charge calculation:
  - Some Atoms need to be excluded for the field calculations.
  - By default, these are all atoms of the residue for which the field is calculated on.
  - Alternatively, these atoms can be defined using -exclude_atoms
    (one line per target, or a single line used for all targets).
- Optional -filter_distance and -filter_angle flags restrict the calculation to frames with a given geometry:
  - e.g. -filter_distance :IAA@O1 :SAM@CE 3.2 keeps frames with a distance <= 3.2 A.
  - e.g. -filter_angle :IAA@O1 :SAM@CE :SAM@SD 160 keeps frames with an angle >= 160 degrees.
  - The kept frames and the measured values are written to <out>_frames.dat.
- The total charges of the system, its segments, the protein, hetero molecules and solvent are printed at the start.
- Optional QM charges can be loaded for the calculation of the field:
  - QM charges are calculated with a separate script (QMChargesTools).
  - If QM charges are to be loaded, the output from QMChargesTools should be used (-use_qm_charges True, -qm_charges, and -qm_dict).
  - This is still experimental and will be added in the future!
  - Contact [adrian.bunzel@bsse.ethz.ch](mailto:adrian.bunzel@bsse.ethz.ch) for early access.
---

**The final fields will be saved into a .pkl file with the following data structure:**

    Fields[field_target][field_component][field_1, field_2, ... field_n]
    
        field_target                    : describe the location at which the field is calculated.
                                          either along a bond or at a specific atom
                                          
        field_component                 : describe the part of the system that causes the field. Components are:
                Total                   : Total field
                Protein                 : Protein field
                Solvent                 : Solvent field
                RSN_1, RSN_2, ... RSN_i : Field of each protein residue
                WAT, Na+, Cl-, etc.     : Field from each type of solvent molecule in the system
                
        field_1, field_2, ... field_n   : List of the fields in each frame of the simulation (MV/cm)
                                          bond targets: field projected onto the bond (signed)
                                          point targets: magnitude of each component's field vector


In [ ]:
#@title Setup FieldTools
# Development version of MDAnalysis: reads .tpr files from GROMACS 2026 (tpx 138),
# which the current release (2.10) cannot. Switch back to "pip install mdanalysis" after 2.11 is released.
print("Installing MDAnalysis (development version), this may take a few minutes.")
!pip install -q "git+https://github.com/MDAnalysis/mdanalysis.git@develop#subdirectory=package" &> /dev/null
!python -c "import MDAnalysis; print('MDAnalysis', MDAnalysis.__version__)"
import os
from google.colab import files
current_directory = os.getcwd()
if "FieldTools" in current_directory.split(os.sep):
  %cd .. 
  !rm -r FieldTools
print("Cloning FieldTools.")
!git clone https://github.com/pjmartel/FieldTools &> /dev/null
%cd FieldTools  

In [ ]:
#@title Prepare input files
field_target = ":40@C7 :40@O71 ; :40@O71" #@param {type:"string"}
target_filename = "data/field_target.dat" #@param {type:"string"}
#@markdown - Defines the position at which the field is to be calculated. 
#@markdown - Either at a point or along a bond. Several targets can be defined by adding additional lines.
#@markdown - Several targets can be defined by adding additional lines.
#@markdown - Only in Google Colab: Separate lines with: ; .
field_target = "\n".join(line.strip() for line in field_target.split(";"))
with open(target_filename, 'w') as file: file.write(field_target)

exclude_mask = ":40@HB2 :40@HB1 :40@CB :40@OG :40@C7 :40@O71 :40@C25 :40@H6 :40@CA :40@HA :264@O :264@H1 :264@H2 ; :40@HB2 :40@HB1 :40@CB :40@OG :40@C7 :40@O71 :40@C25 :40@H6 :40@CA :40@HA :264@O :264@H1 :264@H2" #@param {type:"string"}
exclude_mask_filename = "data/exclude_mask.dat" #@param {type:"string"}
#@markdown - Exclude atoms defines which atoms are not included during field calculations
#@markdown - One line per target (here twice, for targets :40@C7 :40@O71 and :40@O71), or a single line used for all targets
#@markdown - Only in Google Colab: Separate lines with: ; .
exclude_mask = "\n".join(line.strip() for line in exclude_mask.split(";"))
with open(exclude_mask_filename, 'w') as file: file.write(exclude_mask)

nc = "data/KPC.nc" #@param {type:"string"}
#@markdown - Input trajectory (Amber .nc, GROMACS .xtc, ...), leave blank to upload
if nc is None or nc == "":
    print("Upload trajectory file")
    upload_dict = files.upload()
    upload_name = list(upload_dict.keys())[0]
    nc = "data/" + upload_name   # keep the extension, it defines the file format
    with open(nc,"wb") as out: out.write(upload_dict[upload_name])
    print()

parm = "data/KPC.parm7" #@param {type:"string"}
#@markdown - Input topology with charges (Amber .parm7, GROMACS .tpr or .top), leave blank to upload
if parm is None or parm == "":
    print("Upload parameter file")
    upload_dict = files.upload()
    upload_name = list(upload_dict.keys())[0]
    parm = "data/" + upload_name   # keep the extension, it defines the file format
    with open(parm,"wb") as out: out.write(upload_dict[upload_name])
    print()

solvent = "WAT,Na+" #@param {type:"string"}
#@markdown - Comma-separated list of all non-protein molecules (solvent, ions, ligands, etc.) in the system, or auto

TIP4P = True #@param {type:"boolean"}
#@markdown - Check this box if the TIP4P water model is used
#@markdown - Recognizes Amber (O, H1, H2, EPW) and GROMACS (OW, HW1, HW2, MW) 4-point waters

pbc = False #@param {type:"boolean"}
#@markdown - Check this box to use the periodic image of every atom closest to the target (recommended for GROMACS trajectories)

filter_distance = "" #@param {type:"string"}
#@markdown - Optional: only use frames where a distance is below a cutoff, as atom1 atom2 cutoff (A), e.g. :IAA@O1 :SAM@CE 3.2 (or MIN:MAX instead of the cutoff)

filter_angle = "" #@param {type:"string"}
#@markdown - Optional: only use frames where an angle is above a cutoff, as atom1 vertex atom3 cutoff (degrees), e.g. :IAA@O1 :SAM@CE :SAM@SD 160 (or MIN:MAX)

filters = ""
if filter_distance.strip(): filters += "-filter_distance " + filter_distance.strip() + " "
if filter_angle.strip(): filters += "-filter_angle " + filter_angle.strip() + " "

out = "data/field.pkl" #@param {type:"string"}
#@markdown - Output dictionary containing the calculated fields

print(target_filename)
with open(target_filename, 'r') as file: print(file.read())
print()
print(exclude_mask_filename)
with open(exclude_mask_filename, 'r') as file: print(file.read())
print()
print("nc      : ", nc)
print("parm    : ", parm)
print("solvent : ", solvent)
print("TIP4P   : ", TIP4P)
print("pbc     : ", pbc)
print("filters : ", filters or "none")
print("out     : ", out)

In [ ]:
#@title Run the field calculations
!python utils/FieldTools.py -nc "{nc}" -parm "{parm}" -TIP4P "{TIP4P}" -pbc "{pbc}" -solvent "{solvent}" -out "{out}" \
                            -exclude_atoms "{exclude_mask_filename}" -target "{target_filename}" {filters}

In [ ]:
#@title Load the calculated fields
import pickle
with open(out, "rb") as f:
    FIELDS = pickle.load(f)

#-- print out the field keys to give an idea about the data structure
print(FIELDS.keys())
first_target = list(FIELDS)[0]
print(FIELDS[first_target].keys())
print(FIELDS[first_target]['Total'])

In [ ]:
#@title Plot the per-residue fields
import matplotlib.pyplot as plt 
import numpy as np
plt.rcParams["figure.figsize"]=[15,2]
for Target in FIELDS:
    
    # Only protein residues, no Total, Solvent, etc.
    x_keys = [i for i in list(FIELDS[Target]) if len(i.split("_"))==2] 
    
    #get x and y data
    y = np.array([np.mean(FIELDS[Target][x_key]) for x_key in x_keys])
    x = np.array([int(x_key.split("_")[-1]) for x_key in x_keys])
    
    #plot data
    plt.bar(x, y)
    plt.axhline(0,c= "k")
    plt.title(f"Target: {Target}")
    plt.xlabel("Residue Number")
    plt.ylabel("Field (MV/cm)" if "_" in Target else "|Field| (MV/cm)")
    plt.xlim(x.min() - 1, x.max() + 1)
    plt.ylim(-max(abs(y))*1.1, max(abs(y))*1.1)
    plt.show()